# FBref pre-scrapeado → CSV (La Liga / Big 5, temporadas 2018+)

Descarga datos ya recogidos desde el repo `JaseZiv/worldfootballR_data` (GitHub),
**no scrapea FBref** → no pasa por Cloudflare.

Kernel: **R** (ir). Ejecuta celda a celda.

**Decisiones aplicadas:**
- Rango **2018+**: es donde TODAS las tablas (passing, defense, possession, keepers_adv)
  están completas. Antes de 2018 esas tablas no existen y ensuciarían el análisis.
- `SOLO_LALIGA = TRUE` guarda solo La Liga; ponlo en `FALSE` para las Big 5 completas.
- Se descargan las 11 categorías para jugador y equipo.

**Notas de cobertura (ya verificadas):**
- `player_gca` (generación de tiro por jugador) NO está disponible en este dataset →
  saldrá vacío, es esperado.
- `keepers` / `keepers_adv` son para el buscador de porteros, no para jugadores de campo.
- El detalle por partido (shooting logs, advanced match stats) NO existe vía `load_`
  para ninguna liga.

## 0. Instalación (una vez)

In [ ]:
# options(repos = c(CRAN = "https://cloud.r-project.org"))
# install.packages("remotes")
# remotes::install_github("JaseZiv/worldfootballR")   # reinicia el kernel despues
# install.packages("readr")

In [1]:
library(worldfootballR)
library(readr)

## 1. Config
`SOLO_LALIGA`: TRUE = solo La Liga (prefijo de fichero `laliga_`), FALSE = Big 5 (`big5_`).
`SEASON_MIN`: primera temporada a incluir. 2018 es lo recomendado.

In [2]:
SOLO_LALIGA <- TRUE          # TRUE = solo La Liga | FALSE = Big 5 completas
SEASON_MIN  <- 2018          # rango donde TODAS las tablas estan completas
OUT_DIR     <- "data/raw/fbref_r"
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# Filtra a La Liga (si aplica) y a temporadas >= SEASON_MIN
filtrar <- function(df) {
  if (is.null(df) || nrow(df) == 0) return(df)
  if ("Season_End_Year" %in% names(df)) df <- df[df$Season_End_Year >= SEASON_MIN, ]
  if (SOLO_LALIGA && "Comp" %in% names(df)) df <- df[grepl("La Liga", df$Comp), ]
  df
}

save_csv <- function(df, name) {
  df <- filtrar(df)
  if (is.null(df) || nrow(df) == 0) {
    message(sprintf("  [vacio] %s", name)); return(invisible(NULL))
  }
  prefijo <- if (SOLO_LALIGA) "laliga" else "big5"
  path <- file.path(OUT_DIR, sprintf("%s_%s.csv", prefijo, name))
  write_csv(df, path)
  message(sprintf("  [ok] %-24s %5d filas, %2d cols -> %s",
                  name, nrow(df), ncol(df), basename(path)))
}

# Las 11 categorias de FBref advanced season stats
stat_types <- c("standard","shooting","passing","passing_types","gca",
                "defense","possession","playing_time","misc","keepers","keepers_adv")

message(sprintf("Config: %s, temporadas >= %d",
                if (SOLO_LALIGA) "SOLO La Liga" else "Big 5 completas", SEASON_MIN))

Config: SOLO La Liga, temporadas >= 2018



## 2. Tablas de JUGADOR (11 categorías)
`gca` saldrá vacío para jugador (no disponible en el dataset) — es esperado, no es un fallo.

In [3]:
message("=== TABLAS DE JUGADOR ===")

for (st in stat_types) {
  df <- tryCatch(
    load_fb_big5_advanced_season_stats(stat_type = st, team_or_player = "player"),
    error = function(e) { message(sprintf("  [error] player %s: %s", st, conditionMessage(e))); NULL }
  )
  save_csv(df, paste0("player_", st))
}
message("Tablas de jugador completas.")

=== TABLAS DE JUGADOR ===

→ Data last updated 2025-09-18 17:39:32.777853012085 UTC

  [ok] player_standard           5130 filas, 38 cols -> laliga_player_standard.csv

→ Data last updated 2025-09-18 17:40:11.6921770572662 UTC

  [ok] player_shooting           5130 filas, 27 cols -> laliga_player_shooting.csv

→ Data last updated 2025-09-18 17:40:22.6840300559998 UTC

  [ok] player_passing            5130 filas, 37 cols -> laliga_player_passing.csv

→ Data last updated 2025-09-18 17:40:33.5254199504852 UTC

  [ok] player_passing_types      5127 filas, 25 cols -> laliga_player_passing_types.csv

→ Data not available. Check you have the correct stat_type or team_or_player

  [vacio] player_gca

→ Data last updated 2024-10-17 17:40:47.4434180259705 UTC

  [ok] player_defense            4561 filas, 26 cols -> laliga_player_defense.csv

→ Data last updated 2024-10-17 17:41:09.19123697280884 UTC

  [ok] player_possession         4561 filas, 32 cols -> laliga_player_possession.csv

→ Data las

## 3. Tablas de EQUIPO (11 categorías)

In [4]:
message("=== TABLAS DE EQUIPO ===")

for (st in stat_types) {
  df <- tryCatch(
    load_fb_big5_advanced_season_stats(stat_type = st, team_or_player = "team"),
    error = function(e) { message(sprintf("  [error] team %s: %s", st, conditionMessage(e))); NULL }
  )
  save_csv(df, paste0("team_", st))
}
message("Tablas de equipo completas.")

=== TABLAS DE EQUIPO ===

→ Data last updated 2024-10-17 17:41:50.4200720787048 UTC

  [ok] team_standard              320 filas, 36 cols -> laliga_team_standard.csv

→ Data last updated 2024-10-17 17:42:07.84999394416809 UTC

  [ok] team_shooting              320 filas, 24 cols -> laliga_team_shooting.csv

→ Data last updated 2024-10-17 17:42:17.2662301063538 UTC

  [ok] team_passing               320 filas, 32 cols -> laliga_team_passing.csv

→ Data last updated 2024-10-17 17:42:29.4949879646301 UTC

  [ok] team_passing_types         320 filas, 22 cols -> laliga_team_passing_types.csv

→ Data last updated 2024-10-17 17:42:38.5514309406281 UTC

  [ok] team_gca                   320 filas, 23 cols -> laliga_team_gca.csv

→ Data last updated 2024-10-17 17:42:47.9852960109711 UTC

  [ok] team_defense               320 filas, 23 cols -> laliga_team_defense.csv

→ Data last updated 2024-10-17 17:42:56.4818329811096 UTC

  [ok] team_possession            320 filas, 30 cols -> laliga_team_po

## 4. Resultados de partidos
Calendario + resultado + xG por equipo/partido. Base para ratings de equipo.
Cubre desde SEASON_MIN hasta la temporada actual.

In [5]:
message("=== RESULTADOS DE PARTIDOS ===")

paises <- if (SOLO_LALIGA) "ESP" else c("ENG","ESP","ITA","GER","FRA")

resultados <- tryCatch(
  load_match_results(country = paises, gender = "M",
                     season_end_year = SEASON_MIN:2026, tier = "1st"),
  error = function(e) { message("  error: ", conditionMessage(e)); NULL }
)
save_csv(resultados, "match_results")

=== RESULTADOS DE PARTIDOS ===

→ Data last updated 2025-09-16 17:32:02.66567301750183 UTC

  [ok] match_results             3420 filas, 20 cols -> laliga_match_results.csv



## 5. Resumen de lo descargado
Lista los CSV generados con su tamaño, para verificar antes de pasar a Python.

In [6]:
archivos <- list.files(OUT_DIR, pattern = "\\.csv$", full.names = TRUE)
message(sprintf("=== %d CSV en %s ===", length(archivos), OUT_DIR))
for (f in sort(archivos)) {
  kb <- file.info(f)$size / 1024
  message(sprintf("  %-40s %8.0f KB", basename(f), kb))
}

=== 22 CSV en data/raw/fbref_r ===

  laliga_match_results.csv                      689 KB

  laliga_player_defense.csv                     666 KB

  laliga_player_keepers.csv                      53 KB

  laliga_player_keepers_adv.csv                  68 KB

  laliga_player_misc.csv                        662 KB

  laliga_player_passing.csv                    1011 KB

  laliga_player_passing_types.csv               742 KB

  laliga_player_playing_time.csv               1101 KB

  laliga_player_possession.csv                  801 KB

  laliga_player_shooting.csv                    826 KB

  laliga_player_standard.csv                    982 KB

  laliga_team_defense.csv                        50 KB

  laliga_team_gca.csv                            46 KB

  laliga_team_keepers.csv                        48 KB

  laliga_team_keepers_adv.csv                    62 KB

  laliga_team_misc.csv                           47 KB

  laliga_team_passing.csv                        67 KB

  laliga_tea